# Habari phase 3: QLoRA fine-tuning a small LLM for Swahili news classification

This notebook runs on a free Colab T4 GPU (`Runtime > Change runtime type > T4 GPU`).

It measures two things against the classical and AfriBERTa results in the
[habari repo](https://github.com/MaicyMxtim/habari):

1. **Zero-shot**: how well Qwen2.5-1.5B-Instruct classifies Swahili news with no training at all.
2. **QLoRA**: the same model after 4-bit quantized LoRA fine-tuning on the 1,658 training articles.

Both are scored by the repo's own evaluation harness (installed from GitHub below), so every
number is computed by the same code path as the baselines, with bootstrap confidence intervals.
Hyperparameters are fixed in advance rather than tuned, so the test split is still only touched
once per model. Expect the full run to take roughly an hour: most of it is the two evaluation
passes, which generate a label for each of the 476 test articles.

In [ ]:
# kaggle ships torchao 0.10 but peft wants >=0.16; we do not use the
# torchao quantisation path at all, so remove it rather than upgrade torch
%pip uninstall -q -y torchao
%pip install -q -U transformers peft trl bitsandbytes datasets accelerate scikit-learn
%pip install -q git+https://github.com/MaicyMxtim/habari

In [ ]:
from pathlib import Path

from datasets import load_dataset
from habari import evaluate

# Write metrics into the notebook workspace, not into site-packages.
evaluate.RESULTS_DIR = Path("results")

dataset = load_dataset("masakhane/masakhanews", "swa")
labels = sorted(set(dataset["train"]["category"]))
label_to_id = {name: i for i, name in enumerate(labels)}

INSTRUCTION = (
    "Classify this Swahili news article into exactly one category from this list: "
    + ", ".join(labels)
    + ". Answer with the category name only.\n\nArticle:\n"
)


def full_text(row, limit=1500):
    return (row["headline"] + "\n" + row["text"]).strip()[:limit]


print(f"{len(dataset['train'])} train / {len(dataset['test'])} test, labels: {', '.join(labels)}")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,   # transformers 5.x: "dtype", not "torch_dtype"
    device_map={"": 0},
)

In [ ]:
import numpy as np


@torch.no_grad()
def classify(model, rows):
    """Generate a label for each article and map it back to a label id.

    A generation that matches no known label counts as class 0, which is a
    misclassification unless the true class happens to be 0. That penalises
    the model for not following instructions instead of quietly dropping
    those examples.
    """
    predictions = []
    for row in rows:
        messages = [{"role": "user", "content": INSTRUCTION + full_text(row)}]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(model.device)
        out = model.generate(
            **inputs, max_new_tokens=6, do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
        prompt_len = inputs["input_ids"].shape[1]
        answer = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True).lower()
        predictions.append(next((label_to_id[l] for l in labels if l in answer), 0))
    return np.array(predictions)


test = list(dataset["test"])
y_true = np.array([label_to_id[r["category"]] for r in test])

zero_shot_pred = classify(model, test)
metrics = evaluate.evaluate("qwen25_zero_shot", y_true, zero_shot_pred, labels)
evaluate.print_report(metrics)

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer


def to_chat(row):
    return {
        "messages": [
            {"role": "user", "content": INSTRUCTION + full_text(row)},
            {"role": "assistant", "content": row["category"]},
        ]
    }


train_ds = Dataset.from_list([to_chat(r) for r in dataset["train"]])

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
config = SFTConfig(
    output_dir="qlora_out",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=20,
    max_length=256,
    fp16=True,
    gradient_checkpointing=True,   # trades ~30% speed for a lot of memory;
                                   # without it a T4 thrashes and gets slower
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="none",
)
from peft import get_peft_model

# plain LoRA on an fp16 base: no dequantisation and no gradient
# checkpointing, both of which dominated runtime on the T4.
model.enable_input_require_grads()   # required with gradient checkpointing
model = get_peft_model(model, lora)
for _n, _p in model.named_parameters():
    if _p.requires_grad and _p.dtype != torch.float32:
        _p.data = _p.data.float()
model.print_trainable_parameters()

# DIAGNOSTIC: versions and trainable dtypes
import transformers, peft, trl
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| trl", trl.__version__)
print("trainable dtypes:", {str(q.dtype) for q in model.parameters() if q.requires_grad})

trainer = SFTTrainer(model=model, train_dataset=train_ds, args=config)
trainer.train()

In [ ]:
tuned = trainer.model
tuned.eval()

qlora_pred = classify(tuned, test)
metrics = evaluate.evaluate("qwen25_qlora", y_true, qlora_pred, labels)
evaluate.print_report(metrics)

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("qlora_results", "zip", "results")
files.download("qlora_results.zip")

## After the run

The downloaded `qlora_results.zip` contains `qwen25_zero_shot/` and `qwen25_qlora/` folders
with metrics, confusion matrices, and error analyses. Unzip them into the repo's `results/`
directory and commit, then update the results table in the README.